# Intelligent Multi-Modal Weather Intelligence & Climate Decision Support Platform

## Part 2 – Generative AI-powered Weather Research Assistant

### Internship Project

This notebook presents the second phase of the Intelligent Multi-Modal Weather Intelligence and Climate Decision Support Platform. Building upon the Weather Prediction System developed in Part 1, this phase integrates Generative Artificial Intelligence (GenAI) with Retrieval-Augmented Generation (RAG) to create an intelligent Weather Research Assistant capable of answering scientific questions using authoritative climate literature.

Unlike conventional chatbots that rely solely on pretrained knowledge, the proposed assistant retrieves relevant information from trusted climate reports before generating responses. This approach significantly improves factual accuracy, transparency, and domain-specific reasoning while reducing hallucinations commonly associated with large language models.

The assistant utilizes official publications from NASA, NOAA, the Intergovernmental Panel on Climate Change (IPCC), the World Meteorological Organization (WMO), the European Union's Copernicus Climate Change Service, the Organisation for Economic Co-operation and Development (OECD), and the India Meteorological Department (IMD). These reports collectively provide comprehensive scientific evidence regarding weather forecasting, climate change, greenhouse gases, extreme weather events, and environmental sustainability.

The complete workflow consists of document ingestion, text preprocessing, semantic embedding generation, vector database construction using FAISS, retrieval of contextually relevant information, and response generation using Google's Gemini Large Language Model. The resulting system enables users to interact with climate literature through natural language while receiving evidence-based responses grounded in scientific publications.

This notebook demonstrates the practical implementation of Retrieval-Augmented Generation for climate intelligence applications and forms the Generative AI component of the overall Weather Intelligence Platform.

## Objectives

The objectives of this notebook are to:

- Develop a Retrieval-Augmented Generation (RAG) pipeline for climate and weather research.

- Process and index authoritative climate science publications.

- Generate semantic vector embeddings for efficient document retrieval.

- Build a FAISS vector database for similarity search.

- Integrate Google's Gemini Large Language Model for response generation.

- Develop an intelligent Weather Research Assistant capable of answering scientific questions using retrieved evidence.

- Demonstrate practical applications of Generative AI in environmental science and climate research.

- Extend the Weather Prediction System developed in Part 1 into a complete AI-powered Climate Decision Support Platform.

## Workflow

The overall implementation follows the workflow illustrated below.

Weather & Climate Research PDFs

↓

Document Loading

↓

Text Cleaning

↓

Document Chunking

↓

Embedding Generation

↓

FAISS Vector Database

↓

Similarity Search

↓

Relevant Context Retrieval

↓

Gemini Large Language Model

↓

Generative AI Weather Research Assistant

↓

Scientific Answer Generation

# Installing Required Libraries

The implementation requires several open-source libraries for document processing, vector database construction, semantic embedding generation, Retrieval-Augmented Generation (RAG), and interaction with Google's Gemini Large Language Model.

The following cell installs all required dependencies.

In [10]:
!pip install -q \
langchain \
langchain-community \
langchain-google-genai \
langchain-text-splitters \
faiss-cpu \
pypdf \
google-generativeai

# Importing Required Libraries

This section imports all required Python libraries for building the Generative AI-powered Weather Research Assistant.

The implementation utilizes LangChain for orchestrating the Retrieval-Augmented Generation (RAG) workflow, PyPDF for extracting text from climate reports, FAISS for semantic vector search, Google's Gemini Large Language Model for intelligent response generation, and supporting libraries for file handling and document preprocessing.

Together, these libraries enable the development of an AI assistant capable of retrieving scientific information from authoritative climate reports and generating context-aware, evidence-based responses.

In [6]:
import os
import warnings

warnings.filterwarnings("ignore")

import google.generativeai as genai

from google.colab import files

from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import FAISS

from langchain_google_genai import (
    GoogleGenerativeAIEmbeddings,
    ChatGoogleGenerativeAI,
)

import langchain

print("LangChain Version :", langchain.__version__)

print("Libraries imported successfully.")

LangChain Version : 1.3.14
Libraries imported successfully.


# Configuring the Gemini API

Google Gemini serves as the Large Language Model responsible for generating scientifically grounded responses.

Authentication is performed using a valid Gemini API key obtained from Google AI Studio.

In [ ]:
GOOGLE_API_KEY = "API_KEY"

genai.configure(api_key=GOOGLE_API_KEY)

print("Gemini configured successfully.")

Gemini configured successfully.


# Uploading Climate Research Documents

The knowledge base of the Weather Research Assistant consists of authoritative climate and weather publications collected from NASA, NOAA, IPCC, WMO, Copernicus, OECD, IMD, and other trusted organizations.

These reports provide the scientific evidence that will be indexed and queried during Retrieval-Augmented Generation.

In [9]:
uploaded = files.upload()

Saving 5a80ba71f8d83923fb0ffbf238a657a3.pdf to 5a80ba71f8d83923fb0ffbf238a657a3 (2).pdf
Saving 7babf571-en.pdf to 7babf571-en (2).pdf
Saving ESOTC-2025-report_compressed.pdf to ESOTC-2025-report_compressed.pdf
Saving GHG-21_en.pdf to GHG-21_en (2).pdf
Saving IPCC_AR6_SYR_SPM.pdf to IPCC_AR6_SYR_SPM (2).pdf
Saving IPCC_AR6_WGI_SPM.pdf to IPCC_AR6_WGI_SPM (2).pdf
Saving noaa_55885_DS1_compressed.pdf to noaa_55885_DS1_compressed.pdf
Saving UNDRR-GAR2025-web-revised_compressed.pdf to UNDRR-GAR2025-web-revised_compressed.pdf
Saving United-in-Science-2024_en.pdf to United-in-Science-2024_en (2).pdf
Saving WMO-1391-2025_en.pdf to WMO-1391-2025_en (2).pdf


# Loading PDF Documents

Each uploaded PDF is processed using LangChain's document loader.

The textual contents extracted from every page form the document corpus used for semantic indexing and retrieval.

In [10]:
documents = []

for pdf in uploaded.keys():
    loader = PyPDFLoader(pdf)
    documents.extend(loader.load())

print("Total Pages Loaded:", len(documents))

Total Pages Loaded: 1205


In [11]:
documents[0]

Document(metadata={'producer': 'TCPDF 6.7.7 (http://www.tcpdf.org)', 'creator': 'PyPDF', 'creationdate': '2026-07-29T08:19:23+00:00', 'title': 'AIWFB', 'author': 'IMD', 'moddate': '2026-07-29T08:19:23+00:00', 'trapped': '/False', 'source': '5a80ba71f8d83923fb0ffbf238a657a3 (2).pdf', 'total_pages': 29, 'page': 0, 'page_label': '1'}, page_content='2026-07-29\nTime of Issue: 13:45:00 hours IST\n(Mid-Day)\n  ALL INDIA WEATHER SUMMARY AND FORECAST BULLETIN\n  Significant Weather Features\nWeather System: \nThe Deep Depression  over north interior Odisha and adjoining  areas of south Jharkhand & Chhattisgarh moved\nwestwards with a speed of 15 kmph during past 6 hours, and lay centred at 0830 hrs IST of today, the 29th July 2026,\nover central parts of Chhattisgarh & adjoining north interior Odisha, near latitude 21.9°N and longitude 83.3°E, close\nto west-northwest of Raigarh (Chhattisgarh), about 60 km east-southeast of Champa (Chhattisgarh), 80 km northwest\nof Sambalpur (Odisha), 120 km 

# Document Chunking

Scientific reports are typically large and exceed the context limits of language models. Therefore, each report is divided into smaller overlapping chunks using a Recursive Character Text Splitter.

This preprocessing step improves retrieval quality while preserving contextual continuity across adjacent chunks.

In [12]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print("Total Chunks:", len(chunks))

Total Chunks: 4442


In [13]:
print(chunks[0].page_content)

2026-07-29
Time of Issue: 13:45:00 hours IST
(Mid-Day)
  ALL INDIA WEATHER SUMMARY AND FORECAST BULLETIN
  Significant Weather Features
Weather System: 
The Deep Depression  over north interior Odisha and adjoining  areas of south Jharkhand & Chhattisgarh moved
westwards with a speed of 15 kmph during past 6 hours, and lay centred at 0830 hrs IST of today, the 29th July 2026,
over central parts of Chhattisgarh & adjoining north interior Odisha, near latitude 21.9°N and longitude 83.3°E, close
to west-northwest of Raigarh (Chhattisgarh), about 60 km east-southeast of Champa (Chhattisgarh), 80 km northwest
of Sambalpur (Odisha), 120 km east of Bilaspur (Chhattisgarh) and 180 km east-northeast of Raipur (Chhattisgarh). It
is very likely to continue to move nearly westwards across Chhattisgarh and east Madhya Pradesh during next 24
hours.
Weather forecast and Warnings:
Northwest India:
Fairly Widespread to Widespread  rainfall likely over Himachal Pradesh, Jammu-Kashmir-Ladakh-Gilgit-


# Local Semantic Embedding Generation

Generating semantic embeddings using commercial APIs can be limited by request quotas and rate limits, especially when processing large collections of scientific documents.

To overcome this limitation, this project employs the Sentence Transformers framework to generate embeddings locally. The selected transformer model converts each document chunk into a dense numerical representation that captures semantic meaning without requiring external API calls.

Local embedding generation improves scalability, reduces operational cost, eliminates dependency on cloud embedding services, and enables rapid indexing of large document collections.

In [30]:
!pip install -q sentence-transformers

In [31]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# Initializing the Embedding Model

The all-MiniLM-L6-v2 Sentence Transformer is selected due to its balance between computational efficiency and semantic representation quality.

This lightweight transformer generates 384-dimensional embeddings while maintaining strong semantic retrieval performance, making it suitable for Retrieval-Augmented Generation applications.

In [32]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Local embedding model loaded successfully.")

/tmp/ipykernel_25985/160219268.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Local embedding model loaded successfully.


# Building the FAISS Knowledge Base

Every document chunk is transformed into a semantic vector using the Sentence Transformer embedding model.

The generated vectors are indexed using Facebook AI Similarity Search (FAISS), enabling efficient nearest-neighbour retrieval during user interaction.

In [34]:
vector_db = FAISS.from_documents(
    chunks,
    embeddings
)

print("FAISS Vector Database Created Successfully.")

FAISS Vector Database Created Successfully.


# Saving the Knowledge Base

The completed FAISS vector database is stored locally to avoid regenerating embeddings during future notebook executions.

Persisting the vector database significantly reduces startup time and computational overhead.

In [36]:
vector_db.save_local("weather_vector_database")

print("Knowledge Base Saved Successfully.")

Knowledge Base Saved Successfully.


# Reloading the Knowledge Base

The stored vector database can be reloaded whenever required, allowing immediate access to the indexed climate knowledge repository without repeating the embedding generation process.

In [37]:
vector_db = FAISS.load_local(
    "weather_vector_database",
    embeddings,
    allow_dangerous_deserialization=True
)

print("Knowledge Base Loaded Successfully.")

Knowledge Base Loaded Successfully.


# Creating the Semantic Retriever

The retriever performs semantic similarity search over the FAISS vector database.

For every user query, the retriever identifies the most relevant document chunks before forwarding them to the Gemini Large Language Model for response generation. This Retrieval-Augmented Generation strategy ensures that generated responses remain grounded in scientific literature.

In [38]:
retriever = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k":5}
)

print("Retriever Created Successfully.")

Retriever Created Successfully.


# Testing Semantic Retrieval

Before integrating the language model, the retrieval mechanism is evaluated independently.

The following query demonstrates the retriever's ability to locate scientifically relevant passages from the indexed climate literature.

In [39]:
query = "What are the major causes of climate change?"

results = retriever.invoke(query)

print(results[0].page_content)

high confidence that human-induced climate change is the main driver 14 of these changes. Some recent hot extremes 
observed over the past decade would have been extremely unlikely to occur without human influence on the climate 
system. Marine heatwaves have approximately doubled in frequency since the 1980s ( high confidence ), and human 
influence has very likely contributed to most of them since at least 2006.
  {Box 9.2, 11.2, 11.3, 11.9, TS.2.4, TS.2.6, Box TS.10} (Figure SPM.3)
A.3.2  The frequency and intensity of heavy precipitation events have increased since the 1950s over most land area for which 
observational data are sufficient for trend analysis ( high confidence), and human-induced climate change is likely the 
main driver. Human-induced climate change has contributed to increases in agricultural and ecological droughts15 in some 
regions due to increased land evapotranspiration16 (medium confidence).
  {8.2, 8.3, 11.4, 11.6, 11.9, TS.2.6, Box TS.10} (Figure SPM.3)


# Initializing the Gemini Large Language Model

The Gemini Large Language Model serves as the reasoning engine of the Weather Research Assistant.

Unlike traditional chatbots that generate responses solely from pretrained knowledge, Gemini receives scientifically relevant document excerpts retrieved from the FAISS knowledge base before generating an answer. This Retrieval-Augmented Generation (RAG) approach significantly improves factual accuracy, reduces hallucinations, and ensures that responses remain grounded in authoritative climate literature.

In [40]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GOOGLE_API_KEY,
    temperature=0.2
)

print("Gemini LLM initialized successfully.")

Gemini LLM initialized successfully.


# Creating the Retrieval-Augmented Generation (RAG) Pipeline

The Retrieval-Augmented Generation pipeline combines semantic document retrieval with the reasoning capability of the Gemini Large Language Model.

When a user submits a question, the retriever first searches the FAISS knowledge base for the most relevant scientific passages. These retrieved passages are then provided as contextual evidence to Gemini, enabling the model to generate accurate, evidence-based, and context-aware responses.

In [54]:
def ask_weather_assistant(question):

    docs = retriever.invoke(question)

    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = f"""
You are an expert Weather and Climate Research Assistant.

Answer ONLY using the information provided below.

If the answer is not contained in the provided context, clearly state that the information is unavailable in the retrieved documents.

Context:
{context}

Question:
{question}

Answer:
"""

    response = llm.invoke(prompt)

    # Gemini 3.x response
    if isinstance(response.content, list):
        return "".join(
            item["text"] if isinstance(item, dict) else item.text
            for item in response.content
        )

    return response.content

# Testing the Weather Research Assistant

The developed assistant is evaluated using representative scientific questions related to climate change, greenhouse gases, weather forecasting, and environmental sustainability.

For each query, the system retrieves the most relevant scientific evidence from the indexed document collection before generating a final response using Gemini.

In [55]:
llm = ChatGoogleGenerativeAI(
    model="gemini-flash-latest",
    google_api_key=GOOGLE_API_KEY,
    temperature=0.2
)

print("Gemini initialized successfully.")

Gemini initialized successfully.


In [56]:
question = "What are the major causes of climate change?"

answer = ask_weather_assistant(question)

print(answer)

Based on the provided context, the causes/drivers of climate change and disruptions to the climate system include:

* **Human Influence:** Human-induced factors are identified as a main driver of climate change and changes in weather/climate extremes.
* **Greenhouse Gases:** Greenhouse gases are cited as driving climate change, specifically increased $\text{CO}_2$ levels.
* **Land Degradation (Deforestation and Erosion):** Land degradation reduces the Earth's capacity to sequester carbon and releases stored carbon into the atmosphere as $\text{CO}_2$.
* **Loss of Vegetative Cover:** Disrupts the climate system and impacts temperature regulation by altering the albedo (the amount of sunlight reflected) of land surfaces.


In [57]:
question = "What role does the IPCC play in climate research?"

answer = ask_weather_assistant(question)

print(answer)

Based on the provided context, the specific overall organizational mandate or official definition of the IPCC is not explicitly defined. However, the context details the following specific functions and activities performed by the IPCC in climate research:

* **Assessing Scientific Literature:** The IPCC assesses published scientific literature (up to specified cutoff dates) across its Working Groups (covering physical science basis; impacts, adaptation, and vulnerability; and mitigation of climate change).
* **Evaluating Evidence and Confidence:** It evaluates underlying scientific evidence and agreement to formulate key findings as statements of fact or with an assessed level of confidence using IPCC calibrated language (with qualifiers ranging from very low to very high).
* **Publishing Assessment and Special Reports:** It produces Assessment Reports (such as AR5 and AR6) and Special Reports on specific topics (such as *Global Warming of 1.5°C*, *Climate Change and Land*, and *The O

In [58]:
question = "What are the major findings of NASA regarding climate change?"

answer = ask_weather_assistant(question)

print(answer)

Based on the provided documents, information regarding the major findings of NASA on climate change is unavailable.


# Conclusion

This notebook successfully demonstrates the implementation of a Generative AI-powered Weather Research Assistant using Retrieval-Augmented Generation (RAG).

The developed system integrates semantic document retrieval, local embedding generation, FAISS vector indexing, and Google's Gemini Large Language Model to answer scientific questions using authoritative climate literature.

Unlike conventional conversational AI systems, the proposed assistant retrieves evidence from trusted climate reports before generating responses, thereby improving factual accuracy, reducing hallucinations, and enhancing transparency.

The developed assistant complements the Weather Prediction System presented in Part 1, together forming an Intelligent Multi-Modal Weather Intelligence and Climate Decision Support Platform capable of both numerical weather prediction and scientific climate knowledge retrieval.

This implementation demonstrates a practical application of Generative Artificial Intelligence for environmental science, climate research, and intelligent decision support systems.